# 🌊 AquaSign — Stage 1 Training & Evaluation on Google Colab

This notebook runs the underwater hand-gesture cross-domain training experiments.

**Make sure your runtime is set to GPU:**
`Runtime` -> `Change runtime type` -> `T4 GPU`.

In [ ]:
# 1. Verify GPU availability
!nvidia-smi

## 2. Mount Google Drive
Mount your Google Drive to easily load `cache.zip` and save checkpoints/metrics.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Clone Repository or Set Up Project
If you pushed your repo to GitHub, clone it below (replace `<YOUR_GITHUB_REPO_URL>`).  
*(Or if you copied your folder to Drive, simply `%cd /content/drive/MyDrive/DeepSign`)*

In [ ]:
# Option A: Clone from GitHub
# !git clone <YOUR_GITHUB_REPO_URL> DeepSign
# %cd DeepSign

# Install dependencies
!pip install -q -r requirements.txt
!pip install -q pytest

## 4. Unpack Preprocessed Cache
Unzip `cache.zip` from your Google Drive into the working directory.  
*(Make sure `cache.zip` is placed in your Google Drive root or adjust the path below)*

In [ ]:
# Copy and extract cache
!cp /content/drive/MyDrive/cache.zip . 
!unzip -q cache.zip

# Verify cache contents
!ls -lh cache/

## 5. Verify Unit Tests
Run all pytest tests to ensure everything is strictly passing before starting training.

In [ ]:
!PYTHONPATH=. pytest -v tests/

## 6. Train Stage 1 Experiments

### Experiment 1: Pool -> Sea Baseline (No correction, No alignment)
Runs 30 epochs with AdamW, Cosine LR scheduler, and mixed precision.

In [ ]:
!PYTHONPATH=. python -m aquasign.train \
    --config configs/pool2sea.yaml \
    --corr 0 \
    --align 0 \
    --seed 0

### Experiment 2: Pool -> Sea Full Novelty (Color Correction + Domain Alignment)
Enables learnable color correction module initialized to identity + CORAL feature alignment.

In [ ]:
!PYTHONPATH=. python -m aquasign.train \
    --config configs/pool2sea.yaml \
    --corr 1 \
    --align 1 \
    --seed 0

### (Optional) Experiments 3 & 4: Ablation Studies
- Correction only (`--corr 1 --align 0`)
- Alignment only (`--corr 0 --align 1`)

In [ ]:
# Correction only
!PYTHONPATH=. python -m aquasign.train --config configs/pool2sea.yaml --corr 1 --align 0 --seed 0

# Alignment only
!PYTHONPATH=. python -m aquasign.train --config configs/pool2sea.yaml --corr 0 --align 1 --seed 0

## 7. Evaluate Checkpoints & Test-Time AdaBN Adaptation
Evaluate on Target-Test (Sea) with and without AdaBN.

In [ ]:
# Evaluate Baseline
!PYTHONPATH=. python -m aquasign.evaluate --checkpoint results/runs/pool2sea_c0a0_s0/best.pt
!PYTHONPATH=. python -m aquasign.evaluate --checkpoint results/runs/pool2sea_c0a0_s0/best.pt --adabn

# Evaluate Novelty Model
!PYTHONPATH=. python -m aquasign.evaluate --checkpoint results/runs/pool2sea_c1a1_s0/best.pt
!PYTHONPATH=. python -m aquasign.evaluate --checkpoint results/runs/pool2sea_c1a1_s0/best.pt --adabn

## 8. Backup Results & Checkpoints to Google Drive
Save all checkpoints, metrics, and logs back to Google Drive so you never lose them.

In [ ]:
!zip -r results_backup.zip results/
!cp results_backup.zip /content/drive/MyDrive/
print("✅ Saved results_backup.zip to Google Drive!")